# Sales Data Engineering Challenge
250,000 simulated order records. Complete the TODO cells yourself. Follow docs/TASKS.md.

In [ ]:
%pip install -q pyspark==4.0.1

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()  # Choose the original project ZIP
zip_name = next(name for name in uploaded if name.endswith('.zip'))
with zipfile.ZipFile(zip_name) as archive:
    target = Path('/content/sales_challenge').resolve()
    for member in archive.infolist():
        resolved = (target / member.filename).resolve()
        if not resolved.is_relative_to(target):
            raise ValueError('Unsafe archive path')
    archive.extractall(target)
ROOT = target / 'sales-project'
assert (ROOT / 'data/raw').exists()
print('Project:', ROOT)


In [ ]:
from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder.master('local[2]')
         .appName('SalesEngineeringChallenge')
         .config('spark.sql.shuffle.partitions', '8')
         .getOrCreate())
print('Spark version:', spark.version)
print('Master:', spark.sparkContext.master)


In [ ]:
order_schema = 'order_id string, customer_id string, product_id string, order_date string, quantity string, unit_price_cents string, status string, currency string'
raw_orders = spark.read.schema(order_schema).option('header', True).csv(str(ROOT / 'data/raw/orders_part_*.csv'))
customers = spark.read.schema('customer_id string, region string, segment string').option('header', True).csv(str(ROOT / 'data/raw/customers.csv'))
products = spark.read.schema('product_id string, product_name string, category string').option('header', True).csv(str(ROOT / 'data/raw/products.csv'))
assert raw_orders.count() == 250_000
raw_orders.printSchema()
raw_orders.show(5, truncate=False)
print('Customers:', customers.count(), 'Products:', products.count())


## Deduplication
Create deduped_orders. Count exact duplicates removed. Check order_id uniqueness.

In [ ]:
# TODO: implement this task


## Validation
Create typed columns using F.expr("try_cast(quantity as int)") and similar expressions. Validate dates with try_cast(order_date as date). Use dimension joins to detect unknown keys. Split clean_orders and rejected_orders, with rejection reasons.

In [ ]:
# TODO: implement this task


## Enrichment
Join clean_orders with customers and products. Add month and revenue_cents. Assert row count is unchanged.

In [ ]:
# TODO: implement this task


## Reports
Create completed_orders and monthly_summary, region_summary, product_summary. Compare integer-cent totals.

In [ ]:
# TODO: implement this task


## Parquet outputs
Write outputs with mode("overwrite"). Partition only the clean enriched orders by month. Read them back and verify.

In [ ]:
# TODO: implement this task


## Performance
Time the same aggregation with and without cached input. Warm up, repeat, compare medians and correctness. Inspect explain(mode="formatted").

In [ ]:
# TODO: implement this task


## Acceptance checks
Read checks/expected_results.json using json.load. Assert your actual counts and summaries match. Do not just print the reference answers.

In [ ]:
# TODO: implement this task


## Download your outputs
Run this only after writing your results into ROOT / "outputs". Also save this notebook via File > Download > Download .ipynb.

In [ ]:
import shutil
output_dir = ROOT / 'outputs'
assert output_dir.exists(), 'Complete the Parquet output task first.'
archive_path = shutil.make_archive('/content/sales_results', 'zip', output_dir)
files.download(archive_path)
